# RMM: observed EOFs and skill of the hindcasts

## Purpose

Build the observed combined EOFs (OLR, U850, U200) that define the RMM index, check them, project observations and every experiment onto them, and score the model RMM against the observed one by lead time.

## Inputs

EOF training: daily NOAA OLR (E3SM Diags time series, 1986-2021) and ERA5 U850/U200 (daily means of `ERA5/1x1/6hourly`, 1979-2017) under `OBSERVATION_ROOT` (`util/hind_observation_data.py`); the BoM index from `OBSERVATION_ROOT/MJO_INDEX`. Projection: Model: the 2012-2013 AMIP hindcasts under `INPUT_DATA_ROOT/<case>/post/atm/180x360_aave` (experiments and initial states in `configs/hind_experiment_config.py`). Daily FLUT, U850, U200; the same observations from `HISTORY_DAYS` before `START` (the previous-120-day mean needs them).

## Method

EOFs, Wheeler and Hendon (2004): 15S-15N; Feb 29 dropped; anomalies from the 365-day climatology smoothed over `DOY_SMOOTH_DAYS`; the mean of the previous `LOWFREQ_RM_DAYS` days removed; sqrt(cos lat) weighting; each field divided by its space-time std; leading two modes by randomized SVD; PCs divided by their std and scaled so mean(RMM1² + RMM2²) = 1. EOF order and signs are aligned with the BoM RMM index (`BOM_RMM_FILE`) so phases 1-8 keep their usual meaning. The file stores the climatologies and normalisation scalars that every projection (here and `fcst_atm_mjo/02`) reuses. Checks: re-projection of the training data (correlation with the saved RMMs > 0.999), EOF orthogonality, and agreement with the BoM index (RMM1/RMM2 correlation > 0.8 expected; BoM also removes the ENSO signal and uses NCEP winds, so it is not identical).

Projection: the same preprocessing, with the climatology and normalisation from the EOF file. Each run starts on 2012-01-01, so its previous-120-day mean is taken from the observed anomalies of the preceding days followed by its own (Gottschalck et al. 2010); every model day is usable. A rebuilt EOF file recomputes every projection.

Skill on days when the observed amplitude >= `AMP_THRESH` (Rashid et al. 2011): bivariate correlation COR and RMSE of (RMM1, RMM2) without removing means, in a forward running window [L, L + `WINDOW_DAYS`) and in lead bins (lead day = days since 2012-01-01). A climatological forecast has RMSE √2. With one initialization the lead-bin scores rest on few days; read them with `n_samples`.

## Outputs

`data/hind_atm_mjo/rmm/rmm_obs_eof_<train>_climo<smooth>_rm<lowfreq>.nc`, `rmm_projected_<SOURCE>_<period>.nc`, `rmm_skill_running_<period>.nc`, `rmm_skill_leadbins_<period>.csv`; figures in `figure/hind_atm_mjo/rmm/`.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
WORKFLOW_NAME = "hind_atm_mjo"

EXPERIMENTS = ("CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE")
# EOF training
TRAIN_START, TRAIN_END = "1986-01-01", "2017-12-31"
BOM_RMM_FILE = OBSERVATION_ROOT / "MJO_INDEX/rmm.74toRealtime.txt"
LAT_RANGE = (-15.0, 15.0)
DOY_SMOOTH_DAYS = 31
LOWFREQ_RM_DAYS = 120
# projection and skill
START, END = "2012-01-01", "2013-12-31"
HISTORY_DAYS = 150  # observed days before START (>= LOWFREQ_RM_DAYS)
AMP_THRESH = 1.0
WINDOW_DAYS = 120
MAX_LEAD_DAYS = 120
MIN_SAMPLES = 15
FORCE_COMPUTE = False
RECHECK = False  # also check a reused EOF file (re-reads the 1986-2017 training record)
SHOW = True
FIGURE_DPI = 300
USE_DISTRIBUTED = False  # True: local dask.distributed cluster; worker processes read files in parallel
DASK_WORKERS = 8         # threads (threaded scheduler) or worker processes (distributed); bounds memory

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.dask_helpers import compute_all, configure_dask

client = configure_dask(use_distributed=USE_DISTRIBUTED, n_workers=DASK_WORKERS, n_threads=DASK_WORKERS)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, 'rmm', output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(observation_root=OBSERVATION_ROOT)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")

def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Read input

Training fields, opened only to build the EOFs (or with `RECHECK`, to check a saved file), the BoM index, and the 2012-2013 fields. Saved diagnostics are reused: fields are opened lazily only for the sources whose output file is missing (all of them with `FORCE_COMPUTE`), and loaded in the Derive step. A missing or forced EOF file makes every projection stale, so all sources are read then.

In [ ]:
from configs.hind_experiment_config import check_experiments
from util.hind_model_data import open_model_daily
from util.hind_observation_data import open_reference_daily
from util.mjo_rmm_projection import RMM_FIELDS, read_bom_rmm

eof_nc = data_dir / f"rmm_obs_eof_{TRAIN_START[:4]}-{TRAIN_END[:4]}_climo{DOY_SMOOTH_DAYS}_rm{LOWFREQ_RM_DAYS}.nc"
new_eof = FORCE_COMPUTE or not eof_nc.exists()
products = {"FLUT": "NOAA_OLR", "U850": "ERA5_DAILY", "U200": "ERA5_DAILY"}
train = ({v: open_reference_daily(p, v, start=TRAIN_START, end=TRAIN_END, lat_range=(-20.0, 20.0), **ROOTS)
          for v, p in products.items()} if new_eof or RECHECK else {})
bom = read_bom_rmm(BOM_RMM_FILE)

check_experiments(EXPERIMENTS)
period = f"{START[:4]}-{END[:4]}"
SOURCES = ("OBS", *EXPERIMENTS)
paths = {s: data_dir / f"rmm_projected_{s}_{period}.nc" for s in SOURCES}
todo = [s for s in SOURCES if new_eof or not paths[s].exists()]
obs_start = str(np.datetime64(START) - np.timedelta64(HISTORY_DAYS, "D"))
obs_fields = {v: open_reference_daily(p, v, start=obs_start, end=END, lat_range=(-20.0, 20.0), **ROOTS)
              for v, p in products.items()} if todo else {}
model_fields = {s: {v: open_model_daily(s, v, data_root=INPUT_DATA_ROOT, start=START, end=END) for v in RMM_FIELDS}
                for s in todo if s != "OBS"}
print(f"[READ] EOFs: {'training data' if new_eof else 'saved'}; RMM: {', '.join(todo) or 'nothing: every RMM is saved'}")

## 5. Derive

EOFs and training RMMs (when rebuilt), then RMM1, RMM2, amplitude and phase per source; model runs use the observed history for the previous-120-day mean. The tropical band of every field is loaded as float32 (the training precision) in one Dask graph per stage (`compute_all`).

In [ ]:
from util.mjo_rmm_projection import RmmConfig, RmmProjector, compute_rmm_eofs, tropics_subset

cfg = RmmConfig(lat_range=LAT_RANGE, train_start=TRAIN_START, train_end=TRAIN_END,
                doy_smooth_days=DOY_SMOOTH_DAYS, lowfreq_rm_days=LOWFREQ_RM_DAYS)
train = compute_all({v: tropics_subset(da, cfg.lat_range).astype("float32") for v, da in train.items()})
eofs = (compute_rmm_eofs(train, cfg, source="NOAA OLR, ERA5 U850/U200", bom=bom) if new_eof
        else xr.load_dataset(eof_nc))
proj = RmmProjector(eofs)

lazy = {("OBS", v): da for v, da in obs_fields.items()}
lazy |= {(s, v): da for s, f in model_fields.items() for v, da in f.items()}
loaded = compute_all({k: tropics_subset(da, proj.cfg.lat_range).astype("float32") for k, da in lazy.items()})
obs = {v: loaded["OBS", v] for v in obs_fields}
history = {v: f.sel(time=slice(None, np.datetime64(START) - np.timedelta64(1, "D"))) for v, f in obs.items()}
derived = {s: proj.project(obs, s).sel(time=slice(START, END)) if s == "OBS"
           else proj.project({v: loaded[s, v] for v in RMM_FIELDS}, s, history=history) for s in todo}

## 6. Save

Write the newly derived files, then reload every source from disk so the steps below use the same data as a rerun that reuses the files.

In [ ]:
if new_eof:
    eofs.to_netcdf(eof_nc)
    print(f"[WRITE] {eof_nc}")
for s, ds in derived.items():
    ds.to_netcdf(paths[s])
    print(f"[WRITE] {paths[s]}")
eofs = xr.load_dataset(eof_nc)
rmm = {s: xr.load_dataset(p) for s, p in paths.items()}

## 7. Check

Re-projection of the training data, EOF orthogonality, agreement with BoM; run when the EOFs are built (or with `RECHECK`).

In [ ]:
from util.mjo_rmm_projection import verify_rmm_eofs

if not train:
    print("[SKIP] saved EOF file not re-checked (set RECHECK = True)")
else:
    report = verify_rmm_eofs(eofs, train, bom)
    for key, value in report.items():
        print(f"{key:>12}: {value}")
    if report["flags"]:
        print("[WARN] failed checks:", report["flags"])

## 8. Scores

Running-window and lead-bin COR/RMSE of every experiment against OBS.

In [ ]:
from util.mjo_rmm_projection import rmm_leadbin_skill, rmm_running_skill

running = {e: rmm_running_skill(rmm[e], rmm["OBS"], START, window_days=WINDOW_DAYS, max_lead_days=MAX_LEAD_DAYS,
                                amp_thresh=AMP_THRESH, min_samples=MIN_SAMPLES) for e in EXPERIMENTS}
out = data_dir / f"rmm_skill_running_{period}.nc"
xr.concat([ds.expand_dims(experiment=[e]) for e, ds in running.items()], "experiment").to_netcdf(out)
print(f"[WRITE] {out}")
bins = pd.concat([rmm_leadbin_skill(rmm[e], rmm["OBS"], START, amp_thresh=AMP_THRESH).assign(experiment=e)
                  for e in EXPERIMENTS])
out = data_dir / f"rmm_skill_leadbins_{period}.csv"
bins.to_csv(out, index=False)
print(f"[WRITE] {out}")
display(bins.pivot(index="experiment", columns="lead_bin", values="COR").round(2))

## 9. Plot and show

In [ ]:
from util.mjo_rmm_projection import plot_leadbin_skill, plot_rmm_diagnostics, plot_rmm_eofs, plot_running_skill

save(plot_rmm_eofs(eofs), f"{eof_nc.stem}_eofs.png")
save(plot_rmm_diagnostics(eofs, title=f"Observed RMM {TRAIN_START[:4]}-{TRAIN_END[:4]}", bom=bom),
     f"{eof_nc.stem}_diagnostics.png")
save(plot_running_skill(running), f"rmm_skill_running_{period}.png")
save(plot_leadbin_skill(bins), f"rmm_skill_leadbins_{period}.png")
save(plot_rmm_diagnostics(rmm["OBS"], title=f"Observed RMM {period}"), f"rmm_obs_{period}.png")